# Notebook 4. The game step by step (Fig. 4)

Reproduces Fig. 4. Reads no data. Run time about 4 minutes. `QUICK = True` in the first code cell shortens the runs for a test.

In [1]:
QUICK = False
T_RUN = 3000      # the run read by Fig. 4 and the copying analysis of Fig. S2
T_POP = 5000      # the four conditions, the hypothesis tests, the sweeps and Fig. 5; the same seeds,
                  # so the first 3,000 steps of these runs are the runs of Fig. 4

import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
from math import log2

plt.rcParams.update({'figure.dpi': 110, 'font.size': 8, 'axes.spines.top': False,
                     'axes.spines.right': False, 'axes.linewidth': 0.7,
                     'xtick.major.width': 0.7, 'ytick.major.width': 0.7,
                     'legend.frameon': False, 'font.family': 'DejaVu Sans'})

CEIL = log2(100)          # 6.6439 bits, the even-coupling ceiling at N = 100
BA_KAPPA = 10.10          # matched scale-free ensemble, control-entropy paper Table 1
BA_DH_DEG = 0.357         # its degree-weighted compression
BA_RDEG = -0.17           # its degree assortativity
N = 100
COL = {'baseline': '#7A7A7A', 'explore': '#2E7D4E', 'imitate': '#2E5090', 'both': '#A4161A'}
CONDS = {'baseline': dict(pr=0.0, pe=0.0), 'explore': dict(pr=0.0, pe=0.05),
         'imitate': dict(pr=0.3, pe=0.0), 'both': dict(pr=0.3, pe=0.05)}

def seeds(n):
    return list(range(n if not QUICK else max(2, n // 5)))

print('ceiling log2 N =', round(CEIL, 4), 'bits')

ceiling log2 N = 6.6439 bits


## Step 0. The model

In [2]:
import numpy as np
import networkx as nx

LOG2 = np.log(2.0)


# ----------------------------------------------------------------------
# measures (definitions of the second study)
# ----------------------------------------------------------------------
def control_entropy(v):
    """Control entropy in bits of a non-negative vector of shares (Eq. 11)."""
    v = np.asarray(v, float)
    tot = v.sum()
    if tot <= 0:
        return 0.0
    p = v / tot
    p = p[p > 0]
    return float(-(p * np.log2(p)).sum())


def eigen_shares(W):
    """Shares from the leading eigenvector of a symmetric coupling matrix."""
    vals, vecs = np.linalg.eigh(W)
    v = np.abs(vecs[:, -1])
    return v / v.sum()


def adjacency_measures(A, W):
    """Measures read from the unweighted structure, computed as in Table 1 of the
    control-entropy paper: eigenvector weighting on the largest connected component,
    degree assortativity on the whole graph."""
    G = nx.from_numpy_array(A.astype(int))
    deg = A.sum(axis=1)
    s = W.sum(axis=1)
    iu, ju = np.nonzero(np.triu(A, 1))

    def edge_corr(v):
        x, y = np.concatenate([v[iu], v[ju]]), np.concatenate([v[ju], v[iu]])
        return float(np.corrcoef(x, y)[0, 1]) if x.std() > 1e-12 else float('nan')

    r_deg, r_str = edge_corr(deg.astype(float)), edge_corr(s)
    if not nx.is_connected(G):
        G = G.subgraph(max(nx.connected_components(G), key=len)).copy()
    c = nx.eigenvector_centrality_numpy(G)
    v = np.abs(np.array([c[n] for n in G.nodes()]))
    return dict(H_adj=control_entropy(v), r_deg=float(r_deg), r_str=r_str, gc=G.number_of_nodes())


# ----------------------------------------------------------------------
# the five rules, one function each
# ----------------------------------------------------------------------
def ring_lattice(N, k):
    """Regular ring lattice: every element coupled to its k nearest neighbours."""
    return nx.to_numpy_array(nx.watts_strogatz_graph(N, k, 0.0), dtype=bool)


def softmax_allocation(A, beta, shat):
    """Eq 1. Element i splits one unit of coupling over its partners j with weights
    exp(beta_i * shat_j). beta_i = 0 gives even coupling."""
    L = np.where(A, beta[:, None] * shat[None, :], -np.inf)
    L = L - L.max(axis=1, keepdims=True)
    E = np.where(A, np.exp(L), 0.0)
    return E / E.sum(axis=1, keepdims=True)


def commitment(phi0, phi1, C, fm):
    """Eq 2. Commitment of an element, in the linear form of the published model,
    f = phi0 + phi1 * C, truncated to [0, 1 - fm]."""
    return np.clip(phi0 + phi1 * C, 0.0, 1.0 - fm)


def protection(f, c, pmax, chalf):
    """Eq 8. Saturating protection bought by committed capability f * c."""
    x = f * c
    return np.where(x > 0, pmax / (1.0 + chalf / np.maximum(x, 1e-12)), 0.0)


def fermi(dc, s):
    """Eq 10. Probability of copying a partner whose capability is higher by dc."""
    return 1.0 / (1.0 + np.exp(-s * dc))


def logistic(z, g, theta):
    """Response function of an element, used inside the spread (Eq 5)."""
    return 1.0 / (1.0 + np.exp(-g * (z - theta)))


class ElementModel:
    """Network-agent model of element-level exploration and imitation.

    Each element holds
      beta_i        direction of coupling (0 = even coupling, > 0 = toward stronger partners)
      phi0_i, phi1_i commitment strategy, giving f_i = phi0_i + phi1_i * C_i
      c_i           capability, the accumulated stimulation it compares with partners
    Each coupling (i, j) holds a weight w_ij.
    """

    def __init__(self, N=100, k=6, seed=0,
                 pr=0.3, pe=0.05, sigma=0.1, sigma_f=0.01, s=2.0,   # strategy dynamics
                 M=10, rho=0.5, nE=5, switch=None, psi=0.3,        # feedback, two channels
                 gamma=1.0, pn=0.02, pl=0.3, lam=1.0, g=4.0, theta=1.0,   # seeding and spread
                 fm=0.1, pmax=1.0, chalf=0.1, f0=0.1,              # capability and protection
                 eta=0.05, delta=0.01, w0=1.0,                     # consolidation
                 pw=2.0, kmin=2, loss=1.0,                         # pruning and its cost
                 spread=True, two_part=True):
        self.p = dict(N=N, k=k, pr=pr, pe=pe, sigma=sigma, sigma_f=sigma_f, s=s, M=M, rho=rho,
                      nE=nE, switch=switch, psi=psi, gamma=gamma, pn=pn, pl=pl, lam=lam, g=g,
                      theta=theta, fm=fm, pmax=pmax, chalf=chalf, f0=f0, eta=eta, delta=delta,
                      w0=w0, pw=pw, kmin=kmin, loss=loss, spread=spread, two_part=two_part)
        self.rng = np.random.default_rng(seed)
        self.N, self.k = N, k
        self.A = ring_lattice(N, k)
        self.W = self.A.astype(float) * w0
        self.beta = np.zeros(N)
        self.phi0 = np.full(N, f0)
        self.phi1 = np.zeros(N)
        self.c = np.ones(N)
        self.x = np.zeros(N)          # activation of the previous step
        self.v = np.ones(N) / np.sqrt(N)   # leading eigenvector, kept warm
        self.t = 0
        self.n_act = 0.0
        self.n_rounds = 0.0
        self.set_essential(self.rng.choice(N, nE, replace=False))

    # ---------- state helpers ----------
    def set_essential(self, E):
        self.E = np.asarray(E)
        self.isE = np.zeros(self.N, bool)
        self.isE[self.E] = True

    def strengths(self):
        return self.W.sum(axis=1)

    def shat(self):
        s = self.strengths()
        return s / s.max()

    def centrality(self, iters=8):
        """Leading eigenvector of W by power iteration, warm started, scaled to a
        maximum of one. This is the C of the commitment rule (Eq 2)."""
        v = self.v
        for _ in range(iters):
            v = self.W @ v
            n = np.linalg.norm(v)
            if n < 1e-12:
                v = np.ones(self.N) / np.sqrt(self.N)
                break
            v = v / n
        self.v = np.abs(v)
        return self.v / max(self.v.max(), 1e-12)

    def allocation(self):
        return softmax_allocation(self.A, self.beta, self.shat())

    # ---------- one step ----------
    def feedback(self):
        """Eq 3. Two channels of information.
        Process feedback is intrinsic: every element receives afferent input in
        proportion to its own activation in the previous step.
        Result feedback is augmented: M packets, each landing on a task-essential
        element with probability rho and on any element otherwise."""
        p, rng = self.p, self.rng
        u = p['psi'] * self.x.copy()
        if p['M'] > 0:
            to_E = rng.random(p['M']) < p['rho']
            tgt = np.where(to_E, rng.choice(self.E, p['M']), rng.integers(0, self.N, p['M']))
            u = u + np.bincount(tgt, minlength=self.N).astype(float)
        return u

    def spread(self, u, a):
        """Eq 4 and Eq 5. Stimulus is seeded where information arrives and then
        travels along the couplings.
        Seeding: element i is stimulated with probability 1 - (1 - p_n) exp(-gamma u_i),
        so an element fires spontaneously at rate p_n and more often when information
        reaches it.
        Spread: an element already stimulated passes activation to partner j with
        the coupling it allocates, and j responds with probability
        pl * logistic(g (d_j - theta)). Each element activates at most once per step.
        The cascade is run to completion inside the step."""
        p, rng, N = self.p, self.rng, self.N
        act = rng.random(N) < (1.0 - (1.0 - p['pn']) * np.exp(-p['gamma'] * u))   # Eq 4
        if not p['spread']:
            self.n_act, self.n_rounds = float(act.sum()), 0.0
            return act
        frontier = act.copy()
        rounds = 0
        while frontier.any() and rounds < 20:
            # drive reaching each candidate through the allocation of active elements
            d = p['lam'] * self.k * (frontier @ a)
            prob = p['pl'] * logistic(d, p['g'], p['theta'])       # Eq 5
            new = (rng.random(N) < prob) & ~act
            act |= new
            frontier = new
            rounds += 1
        self.n_act, self.n_rounds = float(act.sum()), float(rounds)
        return act

    def step(self):
        p, rng, N = self.p, self.rng, self.N
        if p['switch'] and self.t > 0 and self.t % p['switch'] == 0:
            self.set_essential(rng.choice(N, p['nE'], replace=False))

        C = self.centrality()
        a = self.allocation()                                     # Eq 1
        f = commitment(self.phi0, self.phi1, C, p['fm']) if p['two_part'] else np.zeros(N)  # Eq 2
        u = self.feedback()                                       # Eq 3
        act = self.spread(u, a)                                   # Eq 4, Eq 5
        x = act.astype(float)
        self.x = x

        # Eq 6. Payoff is the stimulation the element gathers. It reaches partners
        # through its own allocation and it integrates in proportion to the coupling
        # it holds, so both the direction and the amount of coupling enter.
        s = self.strengths()
        pay = (s / s.mean()) * (a @ x)
        self.c = pay + np.clip(1.0 - p['fm'] - f, 0.0, None) * self.c   # Eq 7

        # Eq 8 and Eq 9. Commitment protects a coupling against decay.
        P = protection(f, self.c, p['pmax'], p['chalf'])
        Pij = 0.5 * (P[:, None] + P[None, :])
        abar = 0.5 * (a + a.T)
        self.W = np.where(self.A,
                          p['w0'] + (1 - p['delta'] * (1 - Pij)) * (self.W - p['w0'])
                          + p['eta'] * self.k * abar * np.outer(x, x), 0.0)

        # Eq S1. Long-term change: a coupling falls out of use, and unless the
        # capability committed to it holds it, it is lost and replaced elsewhere.
        n_rw = int(p['pw']) + (1 if rng.random() < p['pw'] - int(p['pw']) else 0)
        for _ in range(n_rw):
            self._rewire(P)

        # Eq 10. Imitation along a coupling, synchronous, two independent passes.
        b_old, f0_old, f1_old, c_old = self.beta.copy(), self.phi0.copy(), self.phi1.copy(), self.c.copy()
        for i in np.flatnonzero(rng.random(N) < p['pr']):
            nb = np.flatnonzero(self.A[i])
            j = nb[rng.integers(nb.size)]
            if rng.random() < fermi(c_old[j] - c_old[i], p['s']):
                self.beta[i] = b_old[j]
        if p['two_part']:
            for i in np.flatnonzero(rng.random(N) < p['pr']):
                nb = np.flatnonzero(self.A[i])
                j = nb[rng.integers(nb.size)]
                if rng.random() < fermi(c_old[j] - c_old[i], p['s']):
                    self.phi0[i], self.phi1[i] = f0_old[j], f1_old[j]

        # Eq S2. Exploration of both parts of the strategy.
        ex = rng.random(N) < p['pe']
        self.beta[ex] += rng.normal(0.0, p['sigma'], ex.sum())
        if p['two_part']:
            ex0 = rng.random(N) < p['pe']
            self.phi0[ex0] += rng.normal(0.0, p['sigma_f'], ex0.sum())
            ex1 = rng.random(N) < p['pe']
            self.phi1[ex1] += rng.normal(0.0, p['sigma_f'], ex1.sum())
        self.t += 1
        return x

    def _rewire(self, P):
        """One element drops its weakest coupling, unless protection holds it, and
        attaches to a new partner chosen with exp(beta_i * shat_j)."""
        p, rng = self.p, self.rng
        i = rng.integers(self.N)
        deg = self.A.sum(axis=1)
        nb = np.flatnonzero(self.A[i])
        nb = nb[deg[nb] > p['kmin']]
        if deg[i] <= p['kmin'] or nb.size == 0:
            return
        j = nb[np.argmin(self.W[i, nb])]
        if rng.random() < 0.5 * (P[i] + P[j]):     # the coupling is held
            return
        # the route is lost, and with it the capability the two elements had built
        # through it. With loss = 1 this is the capability loss of the published model.
        self.c[i] *= (1.0 - p['loss'])
        self.c[j] *= (1.0 - p['loss'])
        cand = np.flatnonzero(~self.A[i])
        cand = cand[cand != i]
        if cand.size == 0:
            return
        z = self.beta[i] * self.shat()[cand]
        w = np.exp(z - z.max())
        new = rng.choice(cand, p=w / w.sum())
        self.A[i, j] = self.A[j, i] = False
        self.W[i, j] = self.W[j, i] = 0.0
        self.A[i, new] = self.A[new, i] = True
        self.W[i, new] = self.W[new, i] = p['w0']

    # ---------- measures ----------
    def measures(self, adj=False):
        s = self.strengths()
        deg = self.A.sum(axis=1)
        top = np.argsort(s)[-len(self.E):]
        nbE = self.A[self.E].any(axis=0) & ~self.isE
        tot = s.sum()
        C = self.centrality()
        f = commitment(self.phi0, self.phi1, C, self.p['fm'])
        return dict(H_str=control_entropy(s), H_eig=control_entropy(eigen_shares(self.W)),
                    H_deg=control_entropy(deg), kappa=(deg ** 2).mean() / deg.mean(),
                    beta_mean=self.beta.mean(), beta_var=self.beta.var(),
                    f_mean=float(f.mean()), phi0_mean=float(self.phi0.mean()),
                    phi1_mean=float(self.phi1.mean()), c_mean=float(self.c.mean()),
                    act=self.n_act / self.N, rounds=self.n_rounds,
                    hub_align=self.isE[top].mean(),
                    share_E=s[self.isE].sum() / tot, share_nbE=s[nbE].sum() / tot,
                    **(adjacency_measures(self.A, self.W) if adj else {}))

    def run(self, T, every=25, adj=False):
        rec = {}
        for step in range(T):
            self.step()
            if step % every == 0 or step == T - 1:
                m = self.measures(adj=adj)
                m['t'] = self.t
                for key, v in m.items():
                    rec.setdefault(key, []).append(v)
        return {key: np.array(v) for key, v in rec.items()}


def replicate(T, seeds, adj=True, **kw):
    """Run the model for several seeds and return the final measures."""
    out = []
    for sd in seeds:
        m = ElementModel(seed=sd, **kw)
        for _ in range(T):
            m.step()
        out.append(m.measures(adj=adj))
    return {key: np.array([o[key] for o in out]) for key in out[0]}

In [3]:
from scipy.stats import mannwhitneyu

def cliffs_delta(a, b):
    a, b = np.asarray(a), np.asarray(b)
    gt = sum((x > y) for x in a for y in b)
    lt = sum((x < y) for x in a for y in b)
    return (gt - lt) / (len(a) * len(b))

def boot_median(a, reps=5000, seed=0):
    rng2 = np.random.default_rng(seed)
    a = np.asarray(a)
    d = [np.median(rng2.choice(a, len(a))) for _ in range(reps)]
    return np.percentile(d, [2.5, 97.5])

def boot_diff(a, b, reps=5000, seed=0):
    rng2 = np.random.default_rng(seed)
    a, b = np.asarray(a), np.asarray(b)
    d = [np.median(rng2.choice(a, len(a))) - np.median(rng2.choice(b, len(b))) for _ in range(reps)]
    return np.percentile(d, [2.5, 97.5])

def summary(tag, v, fmt='.3f'):
    v = np.asarray(v, float); lo, hi = boot_median(v)
    print(f'{tag}: median {np.median(v):{fmt}} [{lo:{fmt}}, {hi:{fmt}}], '
          f'mean {v.mean():{fmt}}, sd {v.std(ddof=1):{fmt}}, range {v.min():{fmt}} to {v.max():{fmt}}')

def compare(tag, a, b, unit=''):
    u, p = mannwhitneyu(a, b, alternative='two-sided')
    d = cliffs_delta(a, b)
    lo, hi = boot_diff(a, b)
    diff = np.median(a) - np.median(b)
    print(f'{tag}: U = {u:.1f}, p = {p:.4f}, Cliff delta = {d:+.2f}, '
          f'median difference = {diff:+.3f}{unit} [{lo:+.3f}, {hi:+.3f}]')
    return dict(U=u, p=p, d=d, diff=diff, lo=lo, hi=hi)

def star(p):
    return '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'n.s.'

## Step 1. One run, recorded operation by operation

In [4]:
W_CAP = 500 if T_RUN >= 1000 else T_RUN // 6      # the window of panel E, the last 500 steps

def trace_mechanism(seed=0, T=T_RUN, every=25, keep=(0, 100, T_RUN // 3, T_RUN - 1), **kw):
    global CAP_C, CAP_S
    m = ElementModel(seed=seed, **kw)
    rec, X, snaps = {}, [], {}
    CAP_C, CAP_S = np.zeros(m.N), np.zeros(m.N)
    for t in range(T):
        x = m.step()
        X.append(x.astype(np.uint8))
        if t >= T - W_CAP:                          # capability and strength, summed over the window
            CAP_C += m.c; CAP_S += m.strengths()
        if t % every == 0 or t == T - 1:
            mm = m.measures(adj=True); mm['t'] = m.t
            for key, v in mm.items():
                rec.setdefault(key, []).append(v)
        if t in keep:
            C = m.centrality()
            f = commitment(m.phi0, m.phi1, C, m.p['fm'])
            snaps[t] = dict(A=m.A.copy(), W=m.W.copy(), s=m.strengths().copy(), c=m.c.copy(),
                            beta=m.beta.copy(), f=f.copy(), C=C.copy(), isE=m.isE.copy(),
                            P=protection(f, m.c, m.p['pmax'], m.p['chalf']),
                            act_share=m.n_act / m.N, rounds=m.n_rounds)
    rec = {key: np.array(v) for key, v in rec.items()}
    return m, rec, np.array(X), snaps

one, REC, X, SN = trace_mechanism(seed=0)
CAP_C, CAP_S = CAP_C / W_CAP, CAP_S / W_CAP
T_END, T_MID = T_RUN - 1, T_RUN // 3
print(f'seed 0, {T_RUN} steps, activation kept for {X.shape[0]} steps and {X.shape[1]} elements')
print(f'essential elements: {sorted(one.E.tolist())}')
for t in sorted(SN):
    sn = SN[t]
    deg = sn['A'].sum(1)
    print(f't = {t + 1:>5}: kappa = {(deg ** 2).mean() / deg.mean():.2f}, largest degree {deg.max()}, '
          f'compression on strength shares = {CEIL - control_entropy(sn["s"]):.3f} bits, '
          f'mean preference = {sn["beta"].mean():+.3f}, mean commitment = {sn["f"].mean():.3f}, '
          f'mean capability = {sn["c"].mean():.2f}')

seed 0, 3000 steps, activation kept for 3000 steps and 100 elements
essential elements: [26, 30, 50, 61, 81]
t =     1: kappa = 6.00, largest degree 7, compression on strength shares = 0.000 bits, mean preference = +0.002, mean commitment = 0.100, mean capability = 1.02
t =   101: kappa = 6.27, largest degree 9, compression on strength shares = 0.039 bits, mean preference = +0.037, mean commitment = 0.078, mean capability = 2.52
t =  1001: kappa = 8.74, largest degree 24, compression on strength shares = 0.367 bits, mean preference = +1.898, mean commitment = 0.002, mean capability = 2.71
t =  3000: kappa = 23.49, largest degree 94, compression on strength shares = 1.362 bits, mean preference = +2.912, mean commitment = 0.040, mean capability = 3.40


## Step 2. The structure and its centrality, before and after

In [5]:
def eig_centrality(W):
    vals, vecs = np.linalg.eigh(W)
    v = np.abs(vecs[:, np.argmax(vals)])
    return v / v.max()

C0, C1 = eig_centrality(SN[0]['W']), eig_centrality(SN[T_END]['W'])
print(f'centrality at the start: min {C0.min():.3f}, max {C0.max():.3f}, '
      f'entropy of its shares {control_entropy(C0):.4f} bits of {CEIL:.4f}')
print(f'centrality at the end:   min {C1.min():.3f}, max {C1.max():.3f}, '
      f'entropy of its shares {control_entropy(C1):.4f} bits')
rank_end = np.argsort(C1)[::-1]
print(f'the five most central elements at the end: {rank_end[:5].tolist()}, '
      f'of which essential: {int(SN[T_END]["isE"][rank_end[:5]].sum())}')

centrality at the start: min 0.378, max 1.000, entropy of its shares 6.5758 bits of 6.6439
centrality at the end:   min 0.026, max 1.000, entropy of its shares 6.2797 bits
the five most central elements at the end: [30, 50, 81, 61, 26], of which essential: 5


## Step 3. Seeding and spread

In [6]:
act_share = X.mean(1)
print(f'share of elements activated per step: first 100 steps {act_share[:100].mean():.3f}, '
      f'last 500 steps {act_share[-500:].mean():.3f}, whole run {act_share.mean():.3f}')
print(f'essential elements activated per step, whole run: {X[:, one.isE].mean():.3f}, '
      f'others {X[:, ~one.isE].mean():.3f}')
rounds = REC['rounds']
print(f'rounds of spread per step, recorded every 25 steps: median {np.median(rounds):.0f}, '
      f'range {rounds.min():.0f} to {rounds.max():.0f}')
s_end = SN[T_END]['s']
r_act = np.corrcoef(s_end, X[-500:].mean(0))[0, 1]
print(f'correlation between strength at the end and activation rate over the last 500 steps: {r_act:+.3f}')

share of elements activated per step: first 100 steps 0.415, last 500 steps 0.126, whole run 0.219
essential elements activated per step, whole run: 0.735, others 0.192
rounds of spread per step, recorded every 25 steps: median 3, range 1 to 12
correlation between strength at the end and activation rate over the last 500 steps: +0.706


## Step 4. Payoff, capability and protection

In [7]:
sn = SN[T_END]
r_cs = np.corrcoef(sn['c'], sn['s'])[0, 1]
print(f'end of the run: correlation between capability and strength {r_cs:+.3f}')
print(f'capability of the essential elements {sn["c"][sn["isE"]].mean():.2f} against '
      f'{sn["c"][~sn["isE"]].mean():.2f} for the others')
print(f'protection at the end: mean {sn["P"].mean():.3f}, essential elements {sn["P"][sn["isE"]].mean():.3f}')
fc = np.linspace(0, 2, 200)
print(f'protection curve, Eq. 8: half saturation at f c = {one.p["chalf"]}, maximum {one.p["pmax"]}')

end of the run: correlation between capability and strength +0.776
capability of the essential elements 14.59 against 2.81 for the others
protection at the end: mean 0.495, essential elements 0.849
protection curve, Eq. 8: half saturation at f c = 0.1, maximum 1.0


## Step 5. Imitation under the Fermi rule

In [8]:
def copy_rate(sn, pr=one.p['pr'], s_=one.p['s']):
    A, c = sn['A'], sn['c']
    ii, jj = np.nonzero(A)
    pf = fermi(c[jj] - c[ii], s_)
    # each element draws one partner uniformly among its own
    per_el = np.bincount(ii, weights=pf, minlength=len(c)) / np.maximum(np.bincount(ii, minlength=len(c)), 1)
    return pr * per_el.mean(), c[jj] - c[ii]

for t in sorted(SN):
    rate, dc = copy_rate(SN[t])
    print(f't = {t + 1:>5}: expected copies per element per step {rate:.3f}, '
          f'capability differences across couplings: sd {dc.std():.2f}, share beyond ±1 {np.mean(np.abs(dc) > 1):.2f}')

t =     1: expected copies per element per step 0.150, capability differences across couplings: sd 0.23, share beyond ±1 0.00
t =   101: expected copies per element per step 0.159, capability differences across couplings: sd 1.14, share beyond ±1 0.36
t =  1001: expected copies per element per step 0.184, capability differences across couplings: sd 3.51, share beyond ±1 0.75
t =  3000: expected copies per element per step 0.224, capability differences across couplings: sd 11.80, share beyond ±1 0.87


## Step 6. Exploration and the strategy over practice

In [9]:
for t in sorted(SN):
    b = SN[t]['beta']
    print(f't = {t + 1:>5}: preference mean {b.mean():+.3f}, sd {b.std():.3f}, '
          f'range {b.min():+.2f} to {b.max():+.2f}; commitment mean {SN[t]["f"].mean():.3f}')
print(f'recorded preference at the end of the run: {REC["beta_mean"][-1]:+.3f}, '
      f'variance {REC["beta_var"][-1]:.3f}')

t =     1: preference mean +0.002, sd 0.016, range -0.03 to +0.12; commitment mean 0.100
t =   101: preference mean +0.037, sd 0.260, range -0.39 to +0.61; commitment mean 0.078
t =  1001: preference mean +1.898, sd 0.130, range +1.49 to +2.15; commitment mean 0.002
t =  3000: preference mean +2.912, sd 0.149, range +2.65 to +3.26; commitment mean 0.040
recorded preference at the end of the run: +2.912, variance 0.022


## Step 7. What the structure becomes, thirty runs

In [10]:
NS = seeds(30)
BAND = {}
for name in ('both', 'baseline'):
    rows = []
    for sd in NS:
        m = ElementModel(seed=sd, **CONDS[name])
        r = m.run(T_RUN, every=25, adj=True)
        rows.append(r)
    BAND[name] = rows
for name in ('both', 'baseline'):
    kap = np.array([r['kappa'][-1] for r in BAND[name]])
    dhd = np.array([CEIL - r['H_deg'][-1] for r in BAND[name]])
    rd = np.array([r['r_deg'][-1] for r in BAND[name]])
    print(f'--- {name}, {len(NS)} runs at {T_RUN} steps')
    summary('degree heterogeneity kappa', kap, '.2f')
    summary('compression, degree weighting (bits)', dhd)
    summary('degree assortativity', rd)
print(f'matched scale-free ensemble of paper 2: kappa {BA_KAPPA}, degree-weighted compression {BA_DH_DEG} bits, '
      f'assortativity {BA_RDEG}')

--- both, 30 runs at 3000 steps
degree heterogeneity kappa: median 12.49 [10.39, 15.19], mean 13.22, sd 4.78, range 6.30 to 23.49
compression, degree weighting (bits): median 0.441 [0.348, 0.546], mean 0.437, sd 0.203, range 0.036 to 0.775
degree assortativity: median -0.216 [-0.263, -0.186], mean -0.200, sd 0.104, range -0.357 to 0.069
--- baseline, 30 runs at 3000 steps
degree heterogeneity kappa: median 8.63 [8.54, 8.73], mean 8.63, sd 0.28, range 8.14 to 9.24
compression, degree weighting (bits): median 0.283 [0.278, 0.294], mean 0.285, sd 0.026, range 0.241 to 0.344
degree assortativity: median 0.012 [-0.017, 0.035], mean 0.005, sd 0.061, range -0.185 to 0.117
matched scale-free ensemble of paper 2: kappa 10.1, degree-weighted compression 0.357 bits, assortativity -0.17


## Step 8. Who is offered whom, and where couplings land, thirty runs

In [11]:
KMAX = 101
NWIN = 12                                   # windows of 250 steps over 3,000
W_OFF = 250 if T_RUN >= 1000 else T_RUN // 4
EDGES_D = np.concatenate([[-np.inf], np.linspace(-4, 4, 17), [np.inf]])   # 18 bins, the two ends open

class Logged(ElementModel):
    def __init__(self, *a, **kw):
        super().__init__(*a, **kw)
        self.meet = np.zeros((NWIN, 2)); self.cp_str = np.zeros((NWIN, 2)); self.cp_cap = np.zeros((NWIN, 2))
        self.n_draw = np.zeros(NWIN); self.n_copy = np.zeros(NWIN)
        self.expo = np.zeros((NWIN, KMAX)); self.gain = np.zeros((NWIN, KMAX)); self.loss = np.zeros((NWIN, KMAX))
        self.traj = []

    def win(self):
        return min(NWIN - 1, int(self.t * NWIN / T_RUN))

    def step(self):
        p, rng, N = self.p, self.rng, self.N
        w = self.win()
        self.expo[w] += np.bincount(self.A.sum(axis=1), minlength=KMAX)[:KMAX]
        if p['switch'] and self.t > 0 and self.t % p['switch'] == 0:
            self.set_essential(rng.choice(N, p['nE'], replace=False))
        C = self.centrality()
        a = self.allocation()
        f = commitment(self.phi0, self.phi1, C, p['fm']) if p['two_part'] else np.zeros(N)
        u = self.feedback()
        act = self.spread(u, a)
        x = act.astype(float)
        self.x = x
        s = self.strengths()
        pay = (s / s.mean()) * (a @ x)
        self.c = pay + np.clip(1.0 - p['fm'] - f, 0.0, None) * self.c
        P = protection(f, self.c, p['pmax'], p['chalf'])
        Pij = 0.5 * (P[:, None] + P[None, :])
        abar = 0.5 * (a + a.T)
        self.W = np.where(self.A,
                          p['w0'] + (1 - p['delta'] * (1 - Pij)) * (self.W - p['w0'])
                          + p['eta'] * self.k * abar * np.outer(x, x), 0.0)
        n_rw = int(p['pw']) + (1 if rng.random() < p['pw'] - int(p['pw']) else 0)
        for _ in range(n_rw):
            self._rewire(P)
        b_old, f0_old, f1_old, c_old = self.beta.copy(), self.phi0.copy(), self.phi1.copy(), self.c.copy()
        deg, st = self.A.sum(axis=1), self.strengths()
        for i in np.flatnonzero(rng.random(N) < p['pr']):
            nb = np.flatnonzero(self.A[i])
            j = nb[rng.integers(nb.size)]
            copied = rng.random() < fermi(c_old[j] - c_old[i], p['s'])
            self.n_draw[w] += 1
            if deg[j] != deg[i]: self.meet[w, int(deg[j] > deg[i])] += 1
            if copied:
                self.n_copy[w] += 1
                if st[j] != st[i]: self.cp_str[w, int(st[j] > st[i])] += 1
                if c_old[j] != c_old[i]: self.cp_cap[w, int(c_old[j] > c_old[i])] += 1
                self.beta[i] = b_old[j]
        if p['two_part']:
            for i in np.flatnonzero(rng.random(N) < p['pr']):
                nb = np.flatnonzero(self.A[i])
                j = nb[rng.integers(nb.size)]
                if rng.random() < fermi(c_old[j] - c_old[i], p['s']):
                    self.phi0[i], self.phi1[i] = f0_old[j], f1_old[j]
        ex = rng.random(N) < p['pe']
        self.beta[ex] += rng.normal(0.0, p['sigma'], ex.sum())
        if p['two_part']:
            ex0 = rng.random(N) < p['pe']
            self.phi0[ex0] += rng.normal(0.0, p['sigma_f'], ex0.sum())
            ex1 = rng.random(N) < p['pe']
            self.phi1[ex1] += rng.normal(0.0, p['sigma_f'], ex1.sum())
        self.t += 1
        if self.t % 25 == 0:
            self.traj.append((self.t, self.beta.mean(), self.shat().mean()))
        return x

    def _rewire(self, P):
        p, rng = self.p, self.rng
        i = rng.integers(self.N)
        deg = self.A.sum(axis=1)
        nb = np.flatnonzero(self.A[i])
        nb = nb[deg[nb] > p['kmin']]
        if deg[i] <= p['kmin'] or nb.size == 0:
            return
        j = nb[np.argmin(self.W[i, nb])]
        if rng.random() < 0.5 * (P[i] + P[j]):
            return
        self.c[i] *= (1.0 - p['loss'])
        self.c[j] *= (1.0 - p['loss'])
        cand = np.flatnonzero(~self.A[i])
        cand = cand[cand != i]
        if cand.size == 0:
            return
        z = self.beta[i] * self.shat()[cand]
        wgt = np.exp(z - z.max())
        new = rng.choice(cand, p=wgt / wgt.sum())
        w = self.win()
        self.loss[w, deg[j]] += 1; self.gain[w, deg[new]] += 1
        self.A[i, j] = self.A[j, i] = False
        self.W[i, j] = self.W[j, i] = 0.0
        self.A[i, new] = self.A[new, i] = True
        self.W[i, new] = self.W[new, i] = p['w0']

def offered(m):
    # the differences the imitation step of the step just run was offered, with their weights 1 / k_i
    A = m.A; k = A.sum(1); i, j = np.nonzero(A)
    w = 1.0 / k[i]
    return m.c[j] - m.c[i], w / w.sum()

plain, logged = ElementModel(seed=0), Logged(seed=0)
for _ in range(T_RUN):
    plain.step(); logged.step()
m1, m2 = plain.measures(), logged.measures()
CHECK_LOGGED = all(np.isclose(m1[key], m2[key]) for key in m1)
print('check, the logged run follows the plain model step for step:', CHECK_LOGGED,
      f'(seed 0, mean direction preference {m1["beta_mean"]:.2f} in both)')

check, the logged run follows the plain model step for step: True (seed 0, mean direction preference 2.91 in both)


In [12]:
WINS = {'early': (0, W_OFF), 'late': (T_RUN - W_OFF, T_RUN)}
OFFER = {w: dict(hist=np.zeros(len(EDGES_D) - 1), steps=0, runs=[]) for w in WINS}
RUNS = []
for sd in seeds(30):
    m = Logged(seed=sd)
    acc = {w: dict(hist=np.zeros(len(EDGES_D) - 1), pos=0.0, ge4=0.0, le4=0.0, copy=0.0, n=0) for w in WINS}
    for t in range(T_RUN):
        m.step()
        for w, (t0, t1) in WINS.items():
            if t0 <= t < t1:
                d, wt = offered(m)
                a_ = acc[w]
                a_['hist'] += np.histogram(d, bins=EDGES_D, weights=wt)[0]
                a_['pos'] += wt[d > 0].sum(); a_['ge4'] += wt[d >= 4].sum(); a_['le4'] += wt[d <= -4].sum()
                a_['copy'] += (wt * fermi(d, m.p['s'])).sum(); a_['n'] += 1
    for w, a_ in acc.items():
        OFFER[w]['hist'] += a_['hist']; OFFER[w]['steps'] += a_['n']
        OFFER[w]['runs'].append({q: a_[q] / a_['n'] for q in ('pos', 'ge4', 'le4', 'copy')})
    RUNS.append(dict(expo=m.expo, gain=m.gain, loss=m.loss, kappa=m.measures()['kappa'], beta=m.beta.mean()))
for w in WINS:
    OFFER[w]['share'] = OFFER[w]['hist'] / OFFER[w]['steps']
    for q in ('pos', 'ge4', 'le4', 'copy'):
        v = np.array([r[q] for r in OFFER[w]['runs']])
        OFFER[w][q] = v.mean()                        # pooled over the runs, every run with the same weight
        OFFER[w][q + '_med'] = np.median(v); OFFER[w][q + '_iqr'] = np.percentile(v, [25, 75])
CHECK_SUM = all(np.isclose(OFFER[w]['share'].sum(), 1.0) for w in WINS)
print(f'{len(RUNS)} runs; median degree heterogeneity at the end {np.median([r["kappa"] for r in RUNS]):.2f}, '
      f'median mean direction preference {np.median([r["beta"] for r in RUNS]):.2f}')
for w, (t0, t1) in WINS.items():
    o_ = OFFER[w]
    print(f'{w}, steps {t0 + 1} to {t1}: partner better paid in {o_["pos"]:.3f} of the draws '
          f'(per run median {o_["pos_med"]:.3f} [{o_["pos_iqr"][0]:.3f}, {o_["pos_iqr"][1]:.3f}]), '
          f'ahead by 4 or more in {o_["ge4"]:.3f} (median {o_["ge4_med"]:.3f} [{o_["ge4_iqr"][0]:.3f}, '
          f'{o_["ge4_iqr"][1]:.3f}]), behind by 4 or more in {o_["le4"]:.3f}; mean probability of copying '
          f'per draw {o_["copy"]:.3f}')
    print('   shares of draws by bin, from below -4 to above 4:', np.round(o_['share'], 3).tolist())
print('check, the shares of draws in every window sum to one:', CHECK_SUM)

30 runs; median degree heterogeneity at the end 12.49, median mean direction preference 1.72
early, steps 1 to 250: partner better paid in 0.531 of the draws (per run median 0.531 [0.528, 0.533]), ahead by 4 or more in 0.012 (median 0.011 [0.008, 0.016]), behind by 4 or more in 0.009; mean probability of copying per draw 0.526
   shares of draws by bin, from below -4 to above 4: [0.009, 0.006, 0.01, 0.015, 0.025, 0.04, 0.066, 0.113, 0.183, 0.193, 0.127, 0.079, 0.05, 0.032, 0.02, 0.013, 0.008, 0.012]
late, steps 2751 to 3000: partner better paid in 0.647 of the draws (per run median 0.649 [0.624, 0.684]), ahead by 4 or more in 0.296 (median 0.317 [0.221, 0.384]), behind by 4 or more in 0.066; mean probability of copying per draw 0.645
   shares of draws by bin, from below -4 to above 4: [0.066, 0.015, 0.02, 0.025, 0.031, 0.038, 0.045, 0.052, 0.059, 0.062, 0.058, 0.053, 0.047, 0.041, 0.035, 0.03, 0.025, 0.296]
check, the shares of draws in every window sum to one: True


In [13]:
EARLY, LATE = slice(0, NWIN // 3), slice(2 * NWIN // 3, NWIN)      # first and last 1,000 steps
BINS = [2, 3, 4, 5, 6, 7, 8, 9, 11, 14, 18, 25, 35, 101]
N_EL, L_EL = 100, 300

def rates(win):
    E_ = sum(r['expo'][win].sum(axis=0) for r in RUNS)
    G_ = sum(r['gain'][win].sum(axis=0) for r in RUNS)
    return E_, G_

def binned(E_, X_):
    kk = np.arange(KMAX); out = []
    for lo, hi in zip(BINS[:-1], BINS[1:]):
        sel = (kk >= lo) & (kk < hi)
        if E_[sel].sum() > 2000:
            out.append(((kk[sel] * E_[sel]).sum() / E_[sel].sum(), X_[sel].sum() / E_[sel].sum()))
    return np.array(out)

RATE = {}
for tag, win in [('early', EARLY), ('late', LATE)]:
    E_, G_ = rates(win)
    r_att = G_.sum() / (E_.sum() / N_EL)                 # attachments per step
    kk = np.arange(KMAX); ok = E_ > 2000
    y_ = G_[ok] / E_[ok] / r_att - kk[ok] / (2 * L_EL); x_ = 1 / N_EL - kk[ok] / (2 * L_EL)
    p_fit = (E_[ok] * x_ * y_).sum() / (E_[ok] * x_ * x_).sum()
    RATE[tag] = dict(gain=binned(E_, G_), r=r_att, p=p_fit, even=G_.sum() / E_.sum())
    print(f'{tag} practice: {r_att:.2f} attachments per step; random share p of the copying equation '
          f'fitted to the attachment rate = {p_fit:.2f}')
    g = RATE[tag]['gain']
    print('   attachments per element per 1,000 steps by degree:',
          ', '.join(f'{k_:.1f}: {1000 * v:.1f}' for k_, v in g))

early practice: 1.57 attachments per step; random share p of the copying equation fitted to the attachment rate = 1.00
   attachments per element per 1,000 steps by degree: 2.0: 16.9, 3.0: 16.8, 4.0: 16.7, 5.0: 15.8, 6.0: 14.0, 7.0: 15.3, 8.0: 16.0, 9.3: 16.6, 11.6: 18.2, 14.8: 21.4
late practice: 1.58 attachments per step; random share p of the copying equation fitted to the attachment rate = 0.89
   attachments per element per 1,000 steps by degree: 2.0: 14.1, 3.0: 14.8, 4.0: 15.3, 5.0: 16.0, 6.0: 16.3, 7.0: 16.1, 8.0: 16.4, 9.4: 16.3, 11.8: 16.5, 15.3: 17.1, 20.5: 20.2, 28.3: 26.5, 51.7: 37.4


## Step 9. The style of Fig. 4

In [14]:
import matplotlib as mpl
from matplotlib.colors import Normalize, LinearSegmentedColormap
from matplotlib.lines import Line2D
from matplotlib.patches import FancyArrowPatch, Circle, Arc
from matplotlib.legend_handler import HandlerTuple

plt.rcParams.update({'font.family': 'DejaVu Sans', 'font.size': 7, 'axes.labelsize': 6.4, 'xtick.labelsize': 6.0,
                     'ytick.labelsize': 6.0, 'legend.fontsize': 6.0, 'axes.linewidth': 0.6,
                     'xtick.major.width': 0.6, 'ytick.major.width': 0.6, 'xtick.major.size': 2.4,
                     'ytick.major.size': 0, 'mathtext.fontset': 'dejavusans'})
FIG_W = 6.93
PW_T, PH_T = FIG_W / 4, 2.02              # each operation, inches
PW_B, PH_B = FIG_W / 3, 1.52              # each plot, inches
GAP = 0.06                                # between the two rows
BOX_L = (0.42, 0.33, 1.62, 0.94)
INK, NEUTRAL, LINK = '#1A1A1A', '#37474F', '#C3CBD0'
BLUE, RED, GOLD, GREEN, GREY, NAVY = '#2E5090', '#A4161A', '#C9A227', '#2E7D4E', '#8A8A8A', '#1F3F7A'
_HOT = plt.get_cmap('hot_r')(np.linspace(0.06, 0.94, 256))
_COOL = plt.get_cmap('Blues')(np.linspace(0.92, 0.06, 256))
CMAP = LinearSegmentedColormap.from_list('hot_div', np.vstack([_COOL, _HOT]))
VLIM = 2.7157337518864986                 # the color limit of Figs. 2 and 3
CNORM = Normalize(-VLIM, VLIM)

def colour(w, n):
    b = np.log2(np.asarray(w, float) * n)
    return CMAP(CNORM(np.where(np.abs(b) < 1e-9, 0.0, b)))

def axes_at(fig, ox, oy, box):
    W, H = fig.get_size_inches(); l, b, w, h = box
    return fig.add_axes([(ox + l) / W, (oy + b) / H, w / W, h / H])

def heading(fig, ox, oy, pw, ph, letter, title, sub=None, xc=None):
    W, H = fig.get_size_inches(); y = (oy + ph - 0.19) / H
    fig.text((ox + 0.05) / W, y, letter, fontsize=9.4, fontweight='bold', color='k', ha='left', va='bottom')
    fig.text((ox + (pw / 2 if xc is None else xc)) / W, y, title, fontsize=7.9, color='k', ha='center', va='bottom')
    if sub:
        fig.text((ox + pw / 2) / W, y - 0.02 / H, sub, fontsize=6.2, color='k', ha='center', va='top')

def frame(ax, xt, yt, log_x=False):
    if log_x:
        ax.set_xscale('log')
    ax.set_xlim(xt[0], xt[-1]); ax.set_ylim(yt[0], yt[-1]); ax.set_xticks(xt); ax.set_yticks(yt)
    fmt = mpl.ticker.FuncFormatter(lambda v, _: f'{v:g}')
    ax.xaxis.set_major_formatter(fmt); ax.yaxis.set_major_formatter(fmt)
    ax.xaxis.set_minor_locator(mpl.ticker.NullLocator()); ax.yaxis.set_minor_locator(mpl.ticker.NullLocator())
    for s_ in ('left', 'right'):
        ax.spines[s_].set_visible(False)
    for s_ in ('top', 'bottom'):
        ax.spines[s_].set_visible(True); ax.spines[s_].set_linewidth(0.7)
    ax.grid(True, axis='y', alpha=0.40, linestyle=':', linewidth=0.5, color='gray'); ax.set_axisbelow(True)
    ax.tick_params(axis='y', length=0, pad=1.4)
    ax.tick_params(axis='x', length=2.4, width=0.6, direction='out', pad=1.4)

## Step 10. Panels A to D, one step of the game, as one figure

In [15]:
BW, BH = PW_T - 0.08, 1.70                      # the drawing box of each operation, inches
POS = {'e': (0.84, 1.20), 'i': (0.38, 0.76), 'j': (1.28, 0.76),
       'p': (0.12, 0.32), 'q': (1.46, 0.40), 'b': (1.52, 1.04),
       'a1': (0.50, 1.50), 'a2': (1.18, 1.52), 'a3': (1.42, 1.32), 'a4': (0.22, 1.24)}
EDGES = [('e', 'i'), ('e', 'j'), ('e', 'a1'), ('e', 'a2'), ('e', 'a3'), ('e', 'a4'),
         ('i', 'j'), ('i', 'p'), ('j', 'q'), ('j', 'b')]
STUB = {'p': 235, 'q': 290, 'b': 20, 'a1': 115, 'a2': 65, 'a3': 35, 'a4': 160}
DEG = {n: sum(n in e_ for e_ in EDGES) for n in POS}
NCOL = {n: colour([DEG[n] / (2 * len(EDGES))], len(POS))[0] for n in POS}
RAD = {n: 0.030 + 0.016 * DEG[n] for n in POS}
RG = 0.100                                       # radius of a glyph
SHAT = {'e': 1.0, 'i': 0.50, 'j': 0.67, 'p': 0.17, 'q': 0.17, 'b': 0.17}

def soft(beta, keys):
    z = np.exp(beta * np.array([SHAT[k] for k in keys])); return dict(zip(keys, z / z.sum()))
A_I = soft(0.0, ['e', 'j', 'p'])                 # element i, even allocation
A_J = soft(2.0, ['e', 'i', 'q', 'b'])            # element j, tilted toward the strong

def line(ax, p, q, lw=0.7, color=LINK, ls='-', z=2, cap='round'):
    ax.plot([p[0], q[0]], [p[1], q[1]], color=color, lw=lw, ls=ls, solid_capstyle=cap, zorder=z)

def half(ax, p, q, lw, color=BLUE):
    line(ax, p, ((p[0] + q[0]) / 2, (p[1] + q[1]) / 2), lw=lw, color=color, z=2.5, cap='butt')

def stubs(ax):
    for n, ang in STUB.items():
        (x0, y0), a_ = POS[n], np.radians(ang)
        for k_, al in [(0, 0.9), (1, 0.4)]:
            r0, r1 = RAD[n] + 0.01 + 0.05 * k_, RAD[n] + 0.05 + 0.05 * k_
            ax.plot([x0 + r0 * np.cos(a_), x0 + r1 * np.cos(a_)], [y0 + r0 * np.sin(a_), y0 + r1 * np.sin(a_)],
                    color=LINK, lw=0.7, alpha=al, solid_capstyle='butt', zorder=1)

def node(ax, n, stim=False):
    p = POS[n]
    if stim:
        ax.add_patch(Circle(p, RAD[n] + 0.045, facecolor=GOLD, alpha=0.18, edgecolor='none', zorder=3.5))
        ax.add_patch(Circle(p, RAD[n] + 0.022, facecolor='none', edgecolor=GOLD, lw=1.6, zorder=3.6))
    ax.add_patch(Circle(p, RAD[n], facecolor=NCOL[n], edgecolor=NEUTRAL, lw=0.45, zorder=4))

def glyph(ax, n, cap=0.0, prot=0.0, beta=0.0, side=1):
    p = POS[n]
    if prot > 0:
        ax.add_patch(Circle(p, RG + 0.014 + 0.030 * prot, facecolor=NAVY, edgecolor='none', zorder=3.7))
        ax.add_patch(Circle(p, RG + 0.012, facecolor='white', edgecolor='none', zorder=3.8))
    ax.add_patch(Circle(p, RG, facecolor='white', edgecolor=NEUTRAL, lw=0.55, zorder=4))
    if cap > 0:
        ax.add_patch(Circle(p, RG * 0.80 * np.sqrt(cap), facecolor=GOLD, edgecolor='#8C6D0A', lw=0.3, zorder=4.1))
    if beta > 0:
        span = 90.0 * beta
        ax.add_patch(Arc(p, 2 * RG, 2 * RG, theta1=-90 - span / 2, theta2=-90 + span / 2, color=GREEN, lw=2.0,
                         capstyle='butt', zorder=4.3))
    ax.text(p[0] + side * (RG + 0.07), p[1] + RG + 0.03, n, fontsize=7.6, fontstyle='italic',
            ha='center', va='center', zorder=6)

def arrow(ax, p, q, color=INK, lw=0.8, ms=6, rad=0.0, sA=0, sB=0, z=6, style='-|>'):
    ax.add_patch(FancyArrowPatch(p, q, arrowstyle=style, mutation_scale=ms, lw=lw, color=color,
                                 connectionstyle=f'arc3,rad={rad}', shrinkA=sA, shrinkB=sB, zorder=z))

def zigzag(a, b, n=3, amp=0.0):
    d = b - a; L = np.hypot(*d); v = d / L; nrm = np.array([-v[1], v[0]])
    t = np.linspace(0.0, 1.0, 2 * n + 1)
    off = np.zeros_like(t); off[1:-1:2], off[2:-1:2] = amp, -amp
    return np.array([a + v * L * s_ + nrm * o for s_, o in zip(t, off)])

def drive(ax, n, direction, length=0.30):
    # an input drawn as in Liu, Slotine and Barabasi (2011): a source, a zigzag and an arrow into the element
    d = np.array(direction, float); d /= np.hypot(*d); p = np.array(POS[n]); r = RAD[n] + 0.022
    start, z_end, tip = p + d * (r + length), p + d * (r + 0.10), p + d * (r + 0.02)
    zz = zigzag(start, z_end, n=3, amp=0.016)
    ax.scatter([start[0]], [start[1]], s=7, c=INK, lw=0, zorder=7)
    ax.plot(zz[:, 0], zz[:, 1], color=INK, lw=0.6, solid_joinstyle='miter', zorder=7)
    arrow(ax, tuple(z_end), tuple(tip), lw=0.6, ms=4.5)
    ax.text(start[0] + 0.08, start[1] + 0.01, '$u$', fontsize=6.6, ha='center', va='center')

def wiring(ax, drop=(), add=()):
    for a_, b_ in EDGES:
        if (a_, b_) not in drop:
            line(ax, POS[a_], POS[b_])
    for a_, b_ in add:
        line(ax, POS[a_], POS[b_])

def others(ax, stim=()):
    stubs(ax)
    for n in POS:
        if n not in ('i', 'j'):
            node(ax, n, stim=n in stim)

def draw_ops(fig, oy):
    W, H = fig.get_size_inches()
    def box(k):
        ax = axes_at(fig, k * PW_T, oy, (0.04, -0.08, BW, BH))
        ax.set_xlim(0, BW); ax.set_ylim(0, BH); ax.set_aspect('equal'); ax.axis('off')
        return ax
    def rule(ax, s_):
        ax.text(BW / 2, 0.19, s_, fontsize=6.4, ha='center', va='center', color=INK)
    # A. allocate
    ax = box(0)
    wiring(ax); others(ax)
    for q_ in ('e', 'j', 'p'):
        half(ax, POS['i'], POS[q_], 0.4 + 6.0 * A_I[q_])
    for q_ in ('e', 'i', 'q', 'b'):
        half(ax, POS['j'], POS[q_], 0.4 + 6.0 * A_J[q_])
    glyph(ax, 'i', cap=0.30, side=-1)
    glyph(ax, 'j', cap=0.30, beta=2.0, side=1)
    ax.text(POS['i'][0] + 0.26, POS['i'][1] - 0.22, r'$\beta_i$ = 0', fontsize=6.0, ha='center', va='center', color=GREEN)
    ax.text(POS['j'][0] - 0.26, POS['j'][1] - 0.22, r'$\beta_j$ = 2', fontsize=6.0, ha='center', va='center', color=GREEN)
    rule(ax, r'$a_{ij} = e^{\beta_i \hat{s}_j}\, /\, \sum_l\, e^{\beta_i \hat{s}_l}$')
    heading(fig, 0 * PW_T, oy, PW_T, PH_T, 'A', 'Allocate', 'softmax on partner strength')
    # B. stimulated and paid
    ax = box(1)
    wiring(ax); others(ax, stim=('e',))
    drive(ax, 'e', (0.0, 1.0), length=0.24)
    for n, a_ in [('i', A_I['e']), ('j', A_J['e'])]:
        arrow(ax, POS['e'], POS[n], color=GOLD, lw=0.6 + 4.5 * a_, ms=7, sA=13, sB=11)
    glyph(ax, 'i', cap=0.40, side=-1)
    glyph(ax, 'j', cap=0.80, beta=2.0, side=1)
    rule(ax, r'$\pi_i = (s_i / \langle s \rangle)\, \sum_j\, a_{ij}\, x_j$')
    heading(fig, 1 * PW_T, oy, PW_T, PH_T, 'B', 'Stimulated and paid', 'through its own allocation')
    # C. held or rewired
    ax = box(2)
    wiring(ax, drop=(('i', 'p'), ('e', 'j'))); others(ax)
    line(ax, POS['e'], POS['j'], lw=3.0, color=BLUE, z=2.5)                     # consolidated and held
    line(ax, POS['i'], POS['p'], lw=0.8, color=NEUTRAL, ls=(0, (1.2, 1.6)))     # the weakest, dropped
    mx, my = (POS['i'][0] + POS['p'][0]) / 2, (POS['i'][1] + POS['p'][1]) / 2
    ax.text(mx, my, '×', fontsize=8.0, ha='center', va='center', color=INK, zorder=6)
    arrow(ax, POS['i'], POS['a4'], color=BLUE, lw=1.0, ms=6, sA=9, sB=6)        # attached anew
    glyph(ax, 'i', cap=0.0, side=-1)
    glyph(ax, 'j', cap=0.80, prot=0.9, beta=2.0, side=1)
    ax.text(1.17, 1.06, 'held', fontsize=5.8, ha='center', va='center', color=BLUE, rotation=-40)
    ax.text(0.40, 1.06, 'new', fontsize=5.8, ha='center', va='center', color=BLUE)
    rule(ax, r'$\Pr(i \to l) \propto e^{\beta_i \hat{s}_l}$')
    heading(fig, 2 * PW_T, oy, PW_T, PH_T, 'C', 'Held or rewired', 'unless protection holds it')
    # D. compared and copied
    ax = box(3)
    wiring(ax, drop=(('i', 'p'),), add=(('i', 'a4'),)); others(ax)
    glyph(ax, 'i', cap=0.0, beta=2.0, side=-1)
    glyph(ax, 'j', cap=0.80, prot=0.9, beta=2.0, side=1)
    PI, PJ = POS['i'], POS['j']
    arrow(ax, (PJ[0] - 0.05, PJ[1] - RG - 0.04), (PI[0] + 0.05, PI[1] - RG - 0.04), color=RED, lw=1.2, ms=7, rad=-0.40)
    ax.text((PI[0] + PJ[0]) / 2, 0.365, r'copy $\beta_j$', fontsize=6.2, ha='center', va='center', color=RED)
    ER = RG + 0.06
    q1 = (PI[0] + ER * np.cos(np.radians(178)), PI[1] + ER * np.sin(np.radians(178)))
    q2 = (PI[0] + ER * np.cos(np.radians(238)), PI[1] + ER * np.sin(np.radians(238)))
    ax.add_patch(FancyArrowPatch(q1, q2, arrowstyle='<|-|>', mutation_scale=4.5, lw=0.8, color=GREEN,
                                 connectionstyle='arc3,rad=0.35', shrinkA=0, shrinkB=0, zorder=6))
    ax.text(PI[0] - 0.03, PI[1] - 0.31, 'explore', fontsize=5.8, ha='center', va='center', color=GREEN)
    rule(ax, r'$\varphi(c_j - c_i) = 1\, /\, [1 + e^{-s\,(c_j - c_i)}]$')
    heading(fig, 3 * PW_T, oy, PW_T, PH_T, 'D', 'Compared and copied', 'Fermi rule on capability')
    # arrows from one operation to the next, level with the two elements
    for k_ in range(3):
        x0 = (k_ + 1) * PW_T; yy = (oy - 0.08 + POS['i'][1]) / H
        fig.add_artist(FancyArrowPatch(((x0 - 0.12) / W, yy), ((x0 + 0.12) / W, yy), transform=fig.transFigure,
                                       arrowstyle='-|>', mutation_scale=6, lw=0.8, color=NEUTRAL))

fig = plt.figure(figsize=(FIG_W, PH_T)); fig.set_label('Manuscript Fig 4A-D')
draw_ops(fig, 0.0)
plt.show()
print('panels A to D drawn as one figure; shares in A from the softmax at beta 0 and 2:',
      {k: round(float(v), 2) for k, v in A_I.items()}, {k: round(float(v), 2) for k, v in A_J.items()})

panels A to D drawn as one figure; shares in A from the softmax at beta 0 and 2: {'e': 0.33, 'j': 0.33, 'p': 0.33} {'e': 0.57, 'i': 0.21, 'q': 0.11, 'b': 0.11}


## Step 11. Panels E, F and G, each on its own

In [16]:
def draw_E(fig, ox, oy):
    ax = axes_at(fig, ox, oy, BOX_L)
    dg = SN[T_END]['A'].sum(1)
    o = np.argsort(dg)
    ax.scatter(CAP_S[o], CAP_C[o], s=(1.6 + 46.0 * dg / dg.max())[o] + 3, c=colour(CAP_S / CAP_S.sum(), N)[o],
               edgecolors=NEUTRAL, linewidths=0.3, zorder=3)
    frame(ax, [3, 30, 300], [0, 15, 30], log_x=True)
    ax.set_xlim(2.5, 500)
    ax.set_xlabel('strength  $s_i$', labelpad=1.6); ax.set_ylabel('functional capability  $c_i$', labelpad=2.0)
    ax.text(0.05, 0.90, f'$r$ = {R_CAP:.2f}', transform=ax.transAxes, fontsize=6.0, va='top')
    heading(fig, ox, oy, PW_B, PH_B, 'E', 'The strong are paid')

F_L, F_W = 0.42, 1.46
MID_F = np.concatenate([[-4.36], np.linspace(-3.75, 3.75, 16), [4.36]])   # the end bins set apart
WID_F = np.concatenate([[0.44], np.full(16, 0.46), [0.44]])

def pcol(x):
    return CMAP(0.08 + 0.80 * fermi(x, 2.0))      # the probability of copying, as a color

def draw_F(fig, ox, oy):
    sh_e, sh_l = OFFER['early']['share'], OFFER['late']['share']
    ax = axes_at(fig, ox, oy, (F_L, 0.33, F_W, 0.94))
    ax2 = ax.twinx()
    ax2.bar(MID_F, sh_l, width=WID_F, color=pcol(MID_F), edgecolor='none', zorder=1)
    ok = sh_e > 0.002
    ax2.vlines(MID_F[ok], 0, sh_e[ok], color=NEUTRAL, lw=0.5, zorder=2)
    ax2.scatter(MID_F[ok], sh_e[ok], s=9, c=pcol(MID_F[ok]), edgecolors=NEUTRAL, linewidths=0.35, zorder=3)
    ax2.set_ylim(0, 0.40); ax2.set_xlim(-4.6, 4.6)
    ax2.set_yticks([0, 0.2, 0.4]); ax2.yaxis.set_major_formatter(mpl.ticker.FuncFormatter(lambda v, _: f'{v:g}'))
    ax2.tick_params(axis='y', length=0, pad=1.4, labelsize=6.0)
    for s_ in ax2.spines.values():
        s_.set_visible(False)
    ax2.set_ylabel('share of draws', labelpad=2.5, fontsize=6.4, rotation=90)
    ax.set_zorder(ax2.get_zorder() + 1); ax.patch.set_visible(False)
    dcs = np.linspace(-4.6, 4.6, 300)
    ax.plot(dcs, fermi(dcs, 0.0), color=INK, lw=0.7, ls=(0, (3.2, 2.2)), zorder=3)
    ax.plot(dcs, fermi(dcs, 2.0), color=RED, lw=1.1, zorder=4)
    frame(ax, [-4, 0, 4], [0, 0.5, 1])
    ax.set_xlim(-4.6, 4.6)
    ax.set_xticks([-4.36, 0, 4.36]); ax.set_xticklabels(['≤−4', '0', '≥4'])
    ax.text(3.85, 0.53, 'copying model', fontsize=5.6, ha='right', va='bottom')
    ax.text(1.35, 0.80, '$s$ = 2', fontsize=5.8, color=RED, ha='left', va='center')
    ax.legend(handles=[Line2D([], [], color=NEUTRAL, lw=0.5, marker='o', ms=3.0, mfc=CMAP(0.50), mec=NEUTRAL, mew=0.35),
                       (plt.Rectangle((0, 0), 1, 1, fc=CMAP(0.10), ec='none'),
                        plt.Rectangle((0, 0), 1, 1, fc=CMAP(0.86), ec='none'))],
              labels=['early', 'late'], handler_map={tuple: HandlerTuple(ndivide=None, pad=0)},
              title='differences offered', title_fontsize=5.4, fontsize=5.4, loc='upper left',
              bbox_to_anchor=(0.0, 1.0), handlelength=1.3, handletextpad=0.4, labelspacing=0.2, borderaxespad=0.15,
              frameon=False, alignment='left')
    ax.set_xlabel('capability, partner minus own', labelpad=1.6); ax.set_ylabel('probability of copying', labelpad=2.0)
    heading(fig, ox, oy, PW_B, PH_B, 'F', 'The better paid are copied', xc=F_L + F_W / 2)

G_L, G_W = 0.16, 1.62
EARLY_COL = '#9DB4DE'

def draw_G(fig, ox, oy):
    ax = axes_at(fig, ox, oy, (G_L, 0.33, G_W, 0.94))
    for tag, col, ms_ in [('early', EARLY_COL, 0.8), ('late', BLUE, 1.0)]:
        b = RATE[tag]['gain'] * np.array([1, 1000])
        even = 1000 * RATE[tag]['even']                  # the rate if every element were equally likely
        ax.plot(b[:, 0], b[:, 1], '-', color=col, lw=0.9, zorder=3)
        ax.scatter(b[:, 0], b[:, 1], s=ms_ * (4 + 40.0 * b[:, 0] / 55.0), c=CMAP(CNORM(np.log2(b[:, 1] / even))),
                   edgecolors=col, linewidths=0.5, zorder=4)
    kx = np.linspace(2, 60, 200)
    p_ = RATE['late']['p']
    ax.plot(kx, 1000 * RATE['late']['r'] * (p_ / N_EL + (1 - p_) * kx / (2 * L_EL)), color=INK, lw=0.7,
            ls=(0, (3.2, 2.2)), zorder=2)
    frame(ax, [2, 10, 50], [0, 20, 40], log_x=True)
    ax.set_xlim(70, 1.8)                                 # high degree on the left
    ax.yaxis.tick_right(); ax.yaxis.set_label_position('right')
    ax.set_xlabel('degree  $k$', labelpad=1.6); ax.set_ylabel('attachment rate', labelpad=2.5, rotation=90)
    ax.legend(handles=[Line2D([], [], color=EARLY_COL, lw=0.9, marker='o', ms=3.0, mfc='white', mec=EARLY_COL, mew=0.6),
                       Line2D([], [], color=BLUE, lw=0.9, marker='o', ms=3.0, mfc='white', mec=BLUE, mew=0.6),
                       Line2D([], [], color=INK, lw=0.7, ls=(0, (3.2, 2.2)))],
              labels=['early', 'late', 'copying model'], loc='upper right', fontsize=5.8, handlelength=1.6,
              handletextpad=0.4, labelspacing=0.2, borderaxespad=0.15, frameon=False)
    ax.text(0.03, 0.07, r'$\Pi(k) = p/N + (1-p)\,k/2L$', transform=ax.transAxes, fontsize=6.0, ha='left', va='bottom')
    heading(fig, ox, oy, PW_B, PH_B, 'G', 'Attachment becomes copying', xc=PW_B / 2 + 0.04)

R_CAP = np.corrcoef(np.log(CAP_S), CAP_C)[0, 1]
PANELS = [('E', draw_E), ('F', draw_F), ('G', draw_G)]
for letter, draw in PANELS:
    fig = plt.figure(figsize=(PW_B, PH_B)); fig.set_label('Manuscript Fig 4' + letter)
    draw(fig, 0.0, 0.0)
    plt.show()
print(f'panels E, F and G drawn one by one; E: r = {R_CAP:.3f} between capability and the logarithm of strength')

panels E, F and G drawn one by one; E: r = 0.944 between capability and the logarithm of strength


## Step 12. Fig. 4, assembled

In [17]:
FH = PH_T + GAP + PH_B
fig = plt.figure(figsize=(FIG_W, FH)); fig.set_label('Manuscript Fig 4')
draw_ops(fig, PH_B + GAP)
for k_, (letter, draw) in enumerate(PANELS):
    draw(fig, k_ * PW_B, 0.0)
bold = sorted(t.get_text() for t in fig.texts if t.get_fontweight() == 'bold')
plt.show()
print('Figure 4 drawn: the four operations of one step (A to D) and three plots (E to G)')
print('bold type only in the panel letters:', bold == list('ABCDEFG'))

Figure 4 drawn: the four operations of one step (A to D) and three plots (E to G)
bold type only in the panel letters: True


## Step 13. The values the manuscript takes from this notebook

In [18]:
def put(key, v, fmt='.2f'):
    s = format(v, fmt)
    if float(s) == 0:
        s = s.lstrip('-')
    print(f'{key} = {s}')

print('check, the logged run follows the plain model step for step:', CHECK_LOGGED)
print('check, the shares of draws in every window sum to one:', CHECK_SUM)
put('F4_NRUNS', len(RUNS), 'd')
put('F4_W_CAP', W_CAP, 'd'); put('F4_W_OFF', W_OFF, 'd')
put('F4_R_CAP', R_CAP)
for w, W_ in (('early', 'EARLY'), ('late', 'LATE')):
    o_ = OFFER[w]
    put(f'F4_POS_{W_}', o_['pos']); put(f'F4_GE4_{W_}', o_['ge4']); put(f'F4_LE4_{W_}', o_['le4'])
    put(f'F4_COPY_{W_}', o_['copy'])
    put(f'F4_POS_{W_}_MED', o_['pos_med']); put(f'F4_POS_{W_}_Q1', o_['pos_iqr'][0]); put(f'F4_POS_{W_}_Q3', o_['pos_iqr'][1])
    put(f'F4_GE4_{W_}_MED', o_['ge4_med']); put(f'F4_GE4_{W_}_Q1', o_['ge4_iqr'][0]); put(f'F4_GE4_{W_}_Q3', o_['ge4_iqr'][1])
put('F4_P_EARLY', RATE['early']['p']); put('F4_P_LATE', RATE['late']['p'])

check, the logged run follows the plain model step for step: True
check, the shares of draws in every window sum to one: True
F4_NRUNS = 30
F4_W_CAP = 500
F4_W_OFF = 250
F4_R_CAP = 0.94
F4_POS_EARLY = 0.53
F4_GE4_EARLY = 0.01
F4_LE4_EARLY = 0.01
F4_COPY_EARLY = 0.53
F4_POS_EARLY_MED = 0.53
F4_POS_EARLY_Q1 = 0.53
F4_POS_EARLY_Q3 = 0.53
F4_GE4_EARLY_MED = 0.01
F4_GE4_EARLY_Q1 = 0.01
F4_GE4_EARLY_Q3 = 0.02
F4_POS_LATE = 0.65
F4_GE4_LATE = 0.30
F4_LE4_LATE = 0.07
F4_COPY_LATE = 0.64
F4_POS_LATE_MED = 0.65
F4_POS_LATE_Q1 = 0.62
F4_POS_LATE_Q3 = 0.68
F4_GE4_LATE_MED = 0.32
F4_GE4_LATE_Q1 = 0.22
F4_GE4_LATE_Q3 = 0.38
F4_P_EARLY = 1.00
F4_P_LATE = 0.89
